In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import load_model


In [ ]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

labels_file = r"C:\Users\mayak\DeepSyn\datasets\labels.csv"
model_file  = r"C:\Users\mayak\DeepSyn\src\model_training\checkpoints\final_model.h5"
output_file = "model_predictions.csv"

In [ ]:
model = load_model(model_file)
print("Model input shape:", model.input_shape)

In [ ]:
train_features, val_features, all_train_features, test_features, train_targets, val_targets, all_train_targets, test_targets = load(norm='norm')
print("Train features shape:", train_features.shape)
print("All train features shape:", all_train_features.shape)
print("Test features shape:", test_features.shape)
all_features = np.vstack([all_train_features, test_features])
print("All features shape:", all_features.shape)

In [ ]:
expected_features = model.input_shape[1]
if all_features.shape[1] != expected_features:
    raise ValueError(
        f"Feature mismatch! all_features has {all_features.shape[1]} features, "
        f"model expects {expected_features}.")
print("Feature count matches model input.")

In [ ]:
all_predictions = model.predict(all_features, batch_size=1024).squeeze()
print("Predictions shape:", all_predictions.shape)

In [ ]:
labels_df = pd.read_csv(labels_file)
labels_train = labels_df[labels_df['fold'] != 0]
labels_test = labels_df[labels_df['fold'] == 0]
labels_ordered = pd.concat([labels_train, labels_test], axis=0).reset_index(drop=True)

In [ ]:
if len(labels_ordered) != len(all_predictions):
    raise ValueError(
        f"Row mismatch! labels has {len(labels_ordered)} rows "
        f"but predictions has {len(all_predictions)} values.")

In [ ]:
labels_ordered['ID'] = (
    labels_ordered['drug_a_name'] + "_" +
    labels_ordered['drug_b_name'] + "_" +
    labels_ordered['cell_line'])
labels_ordered['predicted_synergy'] = all_predictions.round(8)
pred_df = labels_ordered[['ID', 'drug_a_name', 'drug_b_name', 'cell_line', 'synergy', 'predicted_synergy']]

In [ ]:
pred_df.to_csv(output_file, index=False)
print(f"Predictions saved to: {output_file}")
print(pred_df.head())